# Cotton benchmark — Step 2a: pre-annotate gold test tiles
Runs `best.pt` (YOLOv5n) over all test tiles, stratifies by boll density,
samples ~2,000 gold tiles, and exports CVAT-ready COCO pre-annotations.

**Before running:** Drive folder `cotton_bench/` must contain
`cotton_test_tiles.zip`, `best.pt`, and `test_classes.csv`.
Runtime: GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
import os
assert os.path.exists(f'{DRIVE}/cotton_test_tiles.zip'), 'zip not found in Drive/cotton_bench'
assert os.path.exists(f'{DRIVE}/best.pt'), 'best.pt not found'
assert os.path.exists(f'{DRIVE}/test_classes.csv'), 'test_classes.csv not found'
print('all inputs found')

In [ ]:
!unzip -q {DRIVE}/cotton_test_tiles.zip -d /content/
import glob
tiles = sorted(glob.glob('/content/test_tiles/*.jpg')) + sorted(glob.glob('/content/test_tiles/*.JPG'))
tiles = sorted(set(tiles))
print(len(tiles), 'tiles')

In [ ]:
!pip install -q ultralytics
import torch
model = torch.hub.load('ultralytics/yolov5', 'custom', path=f'{DRIVE}/best.pt')
model.conf = 0.20
model.iou = 0.45
model.max_det = 3000
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
import json, os
from tqdm import tqdm

preds = {}
BATCH = 32
for i in tqdm(range(0, len(tiles), BATCH)):
    batch = tiles[i:i+BATCH]
    results = model(batch, size=640)
    for path, det in zip(batch, results.xyxy):
        d = det.cpu().numpy()
        preds[os.path.basename(path)] = [
            [float(x1), float(y1), float(x2), float(y2), float(c)]
            for x1, y1, x2, y2, c, _ in d
        ]

with open(f'{DRIVE}/test_predictions.json', 'w') as f:
    json.dump(preds, f)
print('saved', len(preds), 'tile predictions to Drive')

In [ ]:
# Stratified gold sample: by class x density bin
import re, csv, random
random.seed(42)

cls_of_stem = {}
with open(f'{DRIVE}/test_classes.csv') as f:
    for row in csv.DictReader(f):
        cls_of_stem[row['stem']] = row['class']

def stem_of(tile_name):
    return re.split(r'_r\d+_c\d+_x\d+_y\d+', tile_name)[0]

def density_bin(n):
    if n == 0: return '0'
    if n <= 20: return '1-20'
    if n <= 60: return '21-60'
    if n <= 120: return '61-120'
    return '120+'

strata = {}
for tile, boxes in preds.items():
    cls = cls_of_stem.get(stem_of(tile), 'unknown')
    strata.setdefault((cls, density_bin(len(boxes))), []).append(tile)

GOLD_TOTAL = 2000
total = sum(len(v) for v in strata.values())
gold = []
print(f"{'class':<18}{'density':<10}{'avail':<8}{'sampled'}")
for (cls, b), lst in sorted(strata.items()):
    n = min(len(lst), max(20, round(GOLD_TOTAL * len(lst) / total)))
    picked = random.sample(lst, n)
    gold.extend(picked)
    print(f"{cls:<18}{b:<10}{len(lst):<8}{n}")
print('gold total:', len(gold))

In [ ]:
# Package gold tiles + COCO pre-annotations for CVAT
import shutil, zipfile

coco = {'images': [], 'annotations': [],
        'categories': [{'id': 1, 'name': 'cotton_boll'}]}
ann_id = 1
os.makedirs('/content/gold_tiles', exist_ok=True)
for img_id, tile in enumerate(sorted(gold), 1):
    shutil.copy(f'/content/test_tiles/{tile}', f'/content/gold_tiles/{tile}')
    coco['images'].append({'id': img_id, 'file_name': tile,
                           'width': 640, 'height': 640})
    for x1, y1, x2, y2, conf in preds[tile]:
        coco['annotations'].append({
            'id': ann_id, 'image_id': img_id, 'category_id': 1,
            'bbox': [x1, y1, x2 - x1, y2 - y1],
            'area': (x2 - x1) * (y2 - y1), 'iscrowd': 0})
        ann_id += 1

with open('/content/gold_preannotations_coco.json', 'w') as f:
    json.dump(coco, f)

shutil.make_archive(f'{DRIVE}/gold_tiles', 'zip', '/content', 'gold_tiles')
shutil.copy('/content/gold_preannotations_coco.json', f'{DRIVE}/')
with open(f'{DRIVE}/gold_list.txt', 'w') as f:
    f.write('\n'.join(sorted(gold)))
print(f"{len(gold)} gold tiles, {ann_id-1} pre-annotation boxes")
print('Drive now has: gold_tiles.zip, gold_preannotations_coco.json, gold_list.txt, test_predictions.json')

In [ ]:
import random, cv2
import matplotlib.pyplot as plt
random.seed(0)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
dense = [t for t in gold if len(preds[t]) > 120]
sparse = [t for t in gold if 1 <= len(preds[t]) <= 30]
for ax, tile in zip(axes, [random.choice(dense), random.choice(dense),
                           random.choice(sparse) if sparse else random.choice(dense)]):
    img = cv2.cvtColor(cv2.imread(f'/content/test_tiles/{tile}'), cv2.COLOR_BGR2RGB)
    for x1, y1, x2, y2, c in preds[tile]:
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 1)
    ax.imshow(img); ax.set_title(f'{len(preds[tile])} boxes', fontsize=10); ax.axis('off')
plt.tight_layout(); plt.savefig('/content/drive/MyDrive/cotton_bench/sanity_check.png', dpi=120)
plt.show()